In [1]:
from pathlib import Path

UCF_ROOT = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/"
HMDB_ROOT = "/kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/"

# ALT_PKG: update this to wherever your alt package folder lives
ALT_PKG = "/kaggle/input/datasets/darsankrishna12/alt-optionb-code/alt "  # ← set this to your actual alt package path

print("UCF_ROOT :", UCF_ROOT)
print("HMDB_ROOT:", HMDB_ROOT)
print("ALT_PKG  :", ALT_PKG)

UCF_ROOT : /kaggle/input/datasets/matthewjansen/ucf101-action-recognition/
HMDB_ROOT: /kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/
ALT_PKG  : /kaggle/input/datasets/darsankrishna12/alt-optionb-code/alt 


In [2]:
import shutil, os

TARGET = "/kaggle/working/alt"

# Remove previous copied project only
if os.path.exists(TARGET):
    shutil.rmtree(TARGET)

# Copy dataset folder
shutil.copytree(ALT_PKG, TARGET)

# Enter project
os.chdir(TARGET)

print("Working dir:", os.getcwd())
print("Contents:", os.listdir(".")[:10])

Working dir: /kaggle/working/alt
Contents: ['corpus', 'training', 'train.py', 'data', 'utils', 'models', 'eval.py', 'configs', 'Align_Before_Adapt_Leveraging_Entity-to-Region_Alignments_for_Generalizable_Video_Action_Recognition-3.pdf']


In [3]:
import os
import json
import shutil
from pathlib import Path

HMDB_SPLITS_SRC = "/kaggle/input/datasets/darsankrishna12/hmdb-traintestsplit/test_train_splits"
HMDB_SPLITS_DST = "/tmp/hmdb_splits/testTrainMultipleSplits"
OUT_DIR = "/kaggle/working/alt/corpus"
LABEL_OFFSET = 101

os.makedirs(HMDB_SPLITS_DST, exist_ok=True)
for fname in os.listdir(HMDB_SPLITS_SRC):
    if fname.endswith(".txt"):
        shutil.copy(os.path.join(HMDB_SPLITS_SRC, fname), HMDB_SPLITS_DST)
print("HMDB split files copied:", len(os.listdir(HMDB_SPLITS_DST)))

hmdb_classes = sorted({
    p.stem.replace("_test_split1", "")
    for p in Path(HMDB_SPLITS_DST).glob("*_test_split1.txt")
})
hmdb_class_map = {name: i for i, name in enumerate(hmdb_classes)}

def build_hmdb(split_num, out_path, flag):
    entries = []
    for p in sorted(Path(HMDB_SPLITS_DST).glob(f"*_test_split{split_num}.txt")):
        class_name = p.stem.replace(f"_test_split{split_num}", "")
        label = hmdb_class_map[class_name] + LABEL_OFFSET
        with open(p) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 2:
                    continue
                fname, file_flag = parts
                if int(file_flag) == flag:
                    entries.append({
                        "path": f"{class_name}/{fname}",
                        "label": label,
                        "dataset": "hmdb",
                    })
    with open(out_path, "w") as handle:
        json.dump(entries, handle, indent=2)
    print(f"Saved {len(entries)} entries -> {out_path}")

build_hmdb(split_num=1, out_path=f"{OUT_DIR}/hmdb_val_split1.json", flag=2)
build_hmdb(split_num=2, out_path=f"{OUT_DIR}/hmdb_val_split2.json", flag=2)
build_hmdb(split_num=3, out_path=f"{OUT_DIR}/hmdb_val_split3.json", flag=2)
print("HMDB split JSONs generated.")


HMDB split files copied: 153
Saved 1530 entries -> /kaggle/working/alt/corpus/hmdb_val_split1.json
Saved 1530 entries -> /kaggle/working/alt/corpus/hmdb_val_split2.json
Saved 1530 entries -> /kaggle/working/alt/corpus/hmdb_val_split3.json
HMDB split JSONs generated.


In [4]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git",
    "git+https://github.com/facebookresearch/ToMe.git",
    "decord",
], check=True)

import sys, os
from pathlib import Path

ALT_DST = "/kaggle/working/alt"
sys.path.insert(0, ALT_DST)
os.chdir(ALT_DST)
print("CWD:", os.getcwd())

# Quick import + upgraded-code verification
from models.alt import ALT
from training.trainer import Trainer
clip_encoder_path = Path("/kaggle/working/alt/models/clip_encoder.py")
clip_encoder_text = clip_encoder_path.read_text(encoding="utf-8")
print("Imports OK")
print("clip_encoder.py mtime:", clip_encoder_path.stat().st_mtime)
print("Has _merge_tokens:", "def _merge_tokens" in clip_encoder_text)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
CWD: /kaggle/working/alt
Imports OK
clip_encoder.py mtime: 1778406048.0810568
Has _merge_tokens: True


In [5]:
import os
import yaml
from pprint import pprint

CFG_PATH = "/kaggle/working/alt/configs/alt_b16.yaml"
cfg = yaml.safe_load(open(CFG_PATH, "r"))

cfg["clip_model"] = "ViT-B/16"
cfg["corpus_path"] = "/kaggle/working/alt/corpus/corpus_embeddings.pt"
cfg["label_file"] = "/kaggle/working/alt/corpus/all_labels_ordered.json"
cfg["train_json"] = "/kaggle/working/alt/corpus/train_labels.json"
cfg["val_json"] = "/kaggle/working/alt/corpus/val_labels.json"
cfg["data_roots"] = {"ucf": UCF_ROOT, "hmdb": HMDB_ROOT}

cfg["corpus_build_if_missing"] = True
cfg["corpus_rebuild"] = True
cfg["corpus_use_t5_filter"] = True
cfg["corpus_use_lesk"] = False
cfg["corpus_t5_model"] = "t5-small"

# Optional runtime overrides (keep defaults unless experimenting)
# cfg["tome_r"] = 4
# cfg["tome_layers"] = None
# cfg["tau"] = 1.0
# cfg["alignment_topk"] = None
# cfg["early_stopping_patience"] = 3

with open(CFG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config patched:")
pprint(cfg)

print("\nPath checks:")
for k in ["corpus_path", "label_file", "train_json", "val_json"]:
    print(f"  {k}: {cfg[k]} -> exists={os.path.exists(cfg[k])}")


Config patched:
{'adapter_blocks': 4,
 'adapter_lr': 8e-05,
 'alignment_topk': None,
 'backbone_lr': 8e-06,
 'batch_size': 16,
 'clip_model': 'ViT-B/16',
 'corpus_build_if_missing': True,
 'corpus_path': '/kaggle/working/alt/corpus/corpus_embeddings.pt',
 'corpus_rebuild': True,
 'corpus_t5_model': 't5-small',
 'corpus_use_lesk': False,
 'corpus_use_t5_filter': True,
 'data_roots': {'hmdb': '/kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/',
                'ucf': '/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/'},
 'early_stopping_patience': 0,
 'epochs': 50,
 'frame_size': 224,
 'freeze_backbone': True,
 'grad_accum_steps': 2,
 'label_file': '/kaggle/working/alt/corpus/all_labels_ordered.json',
 'log_interval': 50,
 'n_heads': 8,
 'num_classes': 152,
 'num_frames': 8,
 'num_workers': 2,
 'seed': 42,
 'tau': 1.0,
 'tome_layers': None,
 'tome_r': 8,
 'train_json': '/kaggle/working/alt/corpus/train_labels.json',
 'use_temporal_pe': False,
 'val_json': '/kag

In [6]:
import yaml
from data.video_dataset import VideoDataset
from data.transforms import get_train_transforms, get_val_transforms

cfg = yaml.safe_load(open("/kaggle/working/alt/configs/alt_b16.yaml", "r"))

train_ds = VideoDataset(
    cfg["train_json"],
    cfg["data_roots"],
    cfg["num_frames"],
    get_train_transforms(cfg["frame_size"]),
    mode="train",
)
val_ds = VideoDataset(
    cfg["val_json"],
    cfg["data_roots"],
    cfg["num_frames"],
    get_val_transforms(cfg["frame_size"]),
    mode="val",
)

# Verification: dataset creation succeeds without monkey-patch
print("VideoDataset uses built-in resolver:", hasattr(train_ds, "_resolve_path"))
print("train_ds size:", len(train_ds), "val_ds size:", len(val_ds))
print("Resolved sample path:", train_ds._resolve_path(train_ds.entries[0]))


VideoDataset uses built-in resolver: True
train_ds size: 13107 val_ds size: 3783
Resolved sample path: /kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/ApplyEyeMakeup/v_ApplyEyeMakeup_g08_c01.avi


In [ ]:
import json
import subprocess
from collections import defaultdict
from pathlib import Path

# Load HMDB val split (not the combined val_json)
hmdb_val_path = "/kaggle/working/alt/corpus/hmdb_val_split1.json"
val_entries   = json.load(open(hmdb_val_path, "r"))

# Group by label, take first 2 classes, up to 5 videos each
bucket = defaultdict(list)
for e in val_entries:
    bucket[e["label"]].append(e)

selected = []
for label in list(bucket.keys())[:2]:
    selected.extend(bucket[label][:5])

tiny_split = "/kaggle/working/alt/corpus/hmdb_val_tiny.json"
with open(tiny_split, "w") as f:
    json.dump(selected, f)

# Build a dummy checkpoint for the sanity run
import torch
from models.alt import ALT
S = torch.load(cfg["corpus_path"], map_location="cpu").float()
tmp_model = ALT(cfg, S)   # uses cfg, includes ToMe, alignment, adapter
tmp_ckpt = "/kaggle/working/runs/alt_b16/init_sanity.pt"
Path("/kaggle/working/runs/alt_b16").mkdir(parents=True, exist_ok=True)
torch.save({"epoch": 0, "best_acc": 0.0, "model_state": tmp_model.state_dict()}, tmp_ckpt)

print("Sanity split size:", len(selected))
if len(selected) == 0:
    raise RuntimeError("No HMDB samples found for sanity split.")

# Run eval on the tiny split
subprocess.run([
    "python", "/kaggle/working/alt/eval.py",
    "--config",       "/kaggle/working/alt/configs/alt_b16.yaml",
    "--checkpoint",   tmp_ckpt,
    "--split_json",   tiny_split,
    "--label_offset", "101",
    "--n_classes",    "51",
], check=True)
print("Tiny zero-shot sanity check completed.")

In [7]:
import os
required = [
    '/kaggle/working/alt/corpus/corpus_embeddings.pt',
    '/kaggle/working/alt/corpus/all_labels_ordered.json',
    '/kaggle/working/alt/corpus/train_labels.json',
    '/kaggle/working/alt/corpus/val_labels.json',
    '/kaggle/working/alt/corpus/hmdb_val_split1.json',
    '/kaggle/working/alt/models/alt.py',
    '/kaggle/working/alt/models/clip_encoder.py',
    '/kaggle/working/alt/models/alignment.py',
    '/kaggle/working/alt/models/video_adapter.py',
    '/kaggle/working/alt/data/video_dataset.py',
    '/kaggle/working/alt/data/transforms.py',
    '/kaggle/working/alt/training/trainer.py',
    '/kaggle/working/alt/training/loss.py',
    '/kaggle/working/alt/train.py',
    '/kaggle/working/alt/eval.py',
    '/kaggle/working/alt/configs/alt_b16.yaml',
]
for f in required:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

✓  /kaggle/working/alt/corpus/corpus_embeddings.pt
✓  /kaggle/working/alt/corpus/all_labels_ordered.json
✓  /kaggle/working/alt/corpus/train_labels.json
✓  /kaggle/working/alt/corpus/val_labels.json
✓  /kaggle/working/alt/corpus/hmdb_val_split1.json
✓  /kaggle/working/alt/models/alt.py
✓  /kaggle/working/alt/models/clip_encoder.py
✓  /kaggle/working/alt/models/alignment.py
✓  /kaggle/working/alt/models/video_adapter.py
✓  /kaggle/working/alt/data/video_dataset.py
✓  /kaggle/working/alt/data/transforms.py
✓  /kaggle/working/alt/training/trainer.py
✓  /kaggle/working/alt/training/loss.py
✓  /kaggle/working/alt/train.py
✓  /kaggle/working/alt/eval.py
✓  /kaggle/working/alt/configs/alt_b16.yaml


In [8]:
import yaml, torch
from models.alt import ALT

cfg = yaml.safe_load(open('/kaggle/working/alt/configs/alt_b16.yaml', 'r'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

S = torch.load(cfg['corpus_path'], map_location='cpu').float()
model = ALT(cfg, S).to(device).eval()

with torch.no_grad():
    dummy = torch.randn(2, cfg['num_frames'], 3, cfg['frame_size'], cfg['frame_size'], device=device)
    token_feats = model.encoder(dummy)

print('Encoder token shape:', tuple(token_feats.shape))
token_count = token_feats.shape[2]
if cfg.get('tome_r', 0) > 0 and token_count >= 197:
    raise RuntimeError(f'ToMe seems inactive: got {token_count} tokens (expected < 197).')
print('ToMe verification passed.')


100%|███████████████████████████████████████| 335M/335M [00:04<00:00, 70.2MiB/s]


Encoder token shape: (2, 8, 101, 512)
ToMe verification passed.


In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/alt")
os.chdir("/kaggle/working/alt")

import yaml, torch
from models.alt import ALT
from data.video_dataset import VideoDataset
from data.transforms import get_train_transforms, get_val_transforms
from training.trainer import Trainer
from training.loss import build_label_embeddings

cfg = yaml.safe_load(open("configs/alt_b16.yaml", "r"))
# Recommended smoke run before full training:
# cfg["epochs"] = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S = torch.load(cfg["corpus_path"], map_location="cpu").float()
model = ALT(cfg, S)
train_ds = VideoDataset(cfg["train_json"], cfg["data_roots"], cfg["num_frames"], get_train_transforms(cfg["frame_size"]), "train")
val_ds = VideoDataset(cfg["val_json"], cfg["data_roots"], cfg["num_frames"], get_val_transforms(cfg["frame_size"]), "val")
C = build_label_embeddings(cfg["label_file"], device)
torch.cuda.empty_cache()
print("Cleared CUDA cache after label embeddings")

trainer = Trainer(model, cfg, train_ds, val_ds, C, device)
trainer.run(start_epoch=0, output_dir="/kaggle/working/runs/alt_b16", log_dir="/kaggle/working/runs/alt_b16")


In [ ]:
import glob
import torch

candidates = sorted(set(
    glob.glob("/kaggle/input/alt_(tome_hmdb+ucftrain)/runs/alt_b16/latest.pt") +
    glob.glob("/kaggle/input/*/latest.pt") +
    glob.glob("/kaggle/input/**/*latest.pt", recursive=True)
))
print("Found checkpoints:")
for c in candidates:
    print(" -", c)

if candidates:
    ranked = []
    for path in candidates:
        try:
            ck = torch.load(path, map_location="cpu")
            ranked.append((int(ck.get("epoch", -1)), path))
        except Exception as e:
            print(f"Skipping unreadable checkpoint {path}: {e}")
    ranked.sort(key=lambda x: x[0])
    best_epoch, best_path = ranked[-1]
    print(f"Resuming from: {best_path} (epoch={best_epoch})")

    trainer = Trainer(model, cfg, train_ds, val_ds, C, device)
    last_ep = trainer.load_checkpoint(best_path)
    print(f"Loaded checkpoint epoch={last_ep}")
    trainer.run(start_epoch=last_ep + 1, output_dir="/kaggle/working/runs/alt_b16", log_dir="/kaggle/working/runs/alt_b16")
else:
    print("No checkpoint found to resume.")


In [31]:
import os
import shutil
import subprocess
import torch

subprocess.run([
    "python", "/kaggle/working/alt/eval.py",
    "--config", "/kaggle/working/alt/configs/alt_b16.yaml",
    "--checkpoint", "/kaggle/working/runs/alt_b16/best.pt",
    "--split_json", "/kaggle/working/alt/corpus/hmdb_val_split1.json",
    "--label_offset", "101",
    "--n_classes", "51",
], check=True)

torch.cuda.empty_cache()
print("Cleared CUDA cache after evaluation")



Loaded checkpoint (epoch 31, best_acc=0.8726)
hmdb_val_split1.json: 0.6418 (982/1530)

Zero-shot accuracy on split: 0.6418
Cleared CUDA cache after evaluation
Saved: /kaggle/working/best.pt
Saved: /kaggle/working/metrics.csv
Saved: /kaggle/working/alt_b16.yaml


In [9]:
!cp /kaggle/input/datasets/darsankrishna12/eval-2/eval.py /kaggle/working/alt/eval.py

In [12]:
import os
import shutil
import subprocess
import torch

hmdb_splits = [
    "/kaggle/working/alt/corpus/hmdb_val_split1.json",
    "/kaggle/working/alt/corpus/hmdb_val_split2.json",
    "/kaggle/working/alt/corpus/hmdb_val_split3.json",
]

if os.path.exists("/kaggle/working/alt/corpus/hmdb_train_split1.json"):
    print("Note: hmdb_train_split1.json exists but will NOT be used for eval.")

for split_path in hmdb_splits:
    if not os.path.exists(split_path):
        print(f"Missing HMDB split: {split_path}")
        continue
    subprocess.run([
        "python", "/kaggle/working/alt/eval.py",
        "--config", "/kaggle/working/alt/configs/alt_b16.yaml",
        "--checkpoint", "/kaggle/working/runs/alt_b16/best.pt",
        "--split_json", split_path,
        "--label_offset", "101",
        "--n_classes", "51",
] , check=True)

torch.cuda.empty_cache()
print("Cleared CUDA cache after evaluation")

Note: hmdb_train_split1.json exists but will NOT be used for eval.
Loaded checkpoint (epoch 31, best_acc=0.8726)
hmdb_val_split1.json: 0.6418 (982/1530)

Fully Supervised accuracy on split: 0.6418
Loaded checkpoint (epoch 31, best_acc=0.8726)
hmdb_val_split2.json: 0.8948 (1369/1530)

Fully Supervised accuracy on split: 0.8948
Loaded checkpoint (epoch 31, best_acc=0.8726)
hmdb_val_split3.json: 0.8791 (1345/1530)

Fully Supervised accuracy on split: 0.8791
Cleared CUDA cache after evaluation
